In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import xgboost as xgb

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
filepath = '../data/CreditScoring_modeling_data.csv'
full_df = pd.read_csv(filepath)

In [ ]:
full_df = full_df.iloc[:, 1:]

In [ ]:
full_df.head()

,Number_of_units,is_First_time_homeowner_No,is_Property_type_sing,is_Origination_channel_reta,is_Occupancy_status_prim,Single_borrower,is_Loan_purpose_purc,Loan_term,Mortgage_Insurance,is_Origination_channel_corr,is_Loan_purpose_noca,is_Property_type_pud,is_Loan_purpose_cash,is_First_time_homeowner,Credit_Score,OLoan_to_value,OEIR,CLoan_to_value,Debt_to_income,is_Occupancy_status_inve,DFlag
0,1,1,1,1,1,1,1,0.54114,2.035779,0,0,0,0,0,0.307767,1.281180,-0.891898,1.259935,0.486223,0,0
1,1,1,0,0,1,1,1,-1.93188,-0.572111,0,0,1,0,0,0.798065,0.368903,-2.465446,0.342768,-0.497975,0,0
2,1,1,1,0,1,0,0,0.54114,-0.572111,1,0,0,1,0,0.387275,0.368903,0.157134,0.342768,1.689132,0,0
3,1,1,0,0,0,1,1,-1.93188,-0.572111,1,0,0,0,0,0.784814,0.064811,-1.154156,0.037045,0.486223,1,0
4,1,1,0,1,1,1,1,0.54114,-0.572111,0,0,0,0,0,0.639050,0.368903,-0.367382,0.342768,1.033000,0,0


In [ ]:
target = 'DFlag'
X = full_df.drop(columns=[target])
y = full_df[target]

In [ ]:
# Train test split
X_train ,X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

In [ ]:
# Define models
models = {
    'LogisticRegression': LogisticRegression(max_iter=1000),
    'RandomForest': RandomForestClassifier(n_estimators=100, random_state=42),
    'GradientBoosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
    'XGBoost': xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss')
}

# evaluate models with cross validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = {}

for name, model in models.items():
  pipeline = Pipeline([
      ('clf', model)
  ])
  scores = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring='roc_auc')
  results[name] = scores
  print(f"{name} ROC-AUC Mean: {scores.mean():.4f}, Std: {scores.std():.4f}")


LogisticRegression ROC-AUC Mean: 0.7673, Std: 0.0196
RandomForest ROC-AUC Mean: 0.6055, Std: 0.0436
GradientBoosting ROC-AUC Mean: 0.7590, Std: 0.0130


/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [13:44:03] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [13:44:04] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [13:44:05] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [13:44:06] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [13:44:07] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_e

XGBoost ROC-AUC Mean: 0.7057, Std: 0.0118


In [ ]:
#Choose the best model based on Mean AUC
best_model_name = max(results, key=lambda k: results[k].mean())
print(f"\nBest model selected: {best_model_name}")


Best model selected: LogisticRegression


In [ ]:
# Train best model
final_model = Pipeline([
    #('scaler', StandardScaler())
    ('clf', models[best_model_name])
])

final_model.fit(X_train, y_train)
y_pred = final_model.predict(X_test)
y_pred_proba = final_model.predict_proba(X_test)[:, 1]

print("\n Classification Report: \n")
print(classification_report(y_test, y_pred))
print(f"AUC-ROC on Test set: {roc_auc_score(y_test, y_pred_proba):.4f}")


 Classification Report: 

              precision    recall  f1-score   support

           0       0.99      1.00      1.00     19728
           1       0.00      0.00      0.00       128

    accuracy                           0.99     19856
   macro avg       0.50      0.50      0.50     19856
weighted avg       0.99      0.99      0.99     19856

AUC-ROC on Test set: 0.7268


/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
